# Agent Dispatch Module - Manya Mahajan (AI Agent Developer)
**Multi-Agent Emergency Response System - Victoria, Australia**

This notebook implements the dispatch system that determines which emergency services to send to an incident, resolves the nearest real facility using geographic data, and calculates the fastest route using a road network routing engine.

---

### Contents
1. Load and Validate Datasets
   - 1.1 Routing Engine Setup
2. Emergency Types and Dispatch Table
3. Nearest-Unit Lookup
4. Base Agent Class
5. Specialised Agents - Ambulance, Fire, Police
6. Area Context - Pedestrian and Transport Awareness
7. Agent Pool and Central Dispatch Function
   - 7.1 Expanded Agent Pool
   - 7.2 Concurrent Incident Stress Test
8. Real Victorian Crash Incidents through Dispatch
9. Full System Test - All 10 Emergency Types
10. Routing Integration Verification
11. Summary

---
## 1 - Load and Validate Datasets

Before any dispatch logic runs the system needs reliable geographic data. This cell loads all eight CSV datasets from the project GitHub repository, standardises column names, converts coordinates to numeric types, and filters hospitals down to emergency-capable facilities only.

A quality check confirms row counts, coordinate ranges and null values across every dataset before any dispatch logic runs.

In [3]:
import pandas as pd
import os
import math
import warnings
from datetime import datetime
import zipfile, io, requests

warnings.filterwarnings('ignore')

# File paths
base_url = "https://raw.githubusercontent.com/Chameleon-company/MOP-Code/master/datascience/usecases/DEPENDENCIES/Project4_Multi_Agent_Emergency_Response_System_Datasets"

hospitals       = pd.read_csv(f"{base_url}/hospitals.csv")
fire_stations   = pd.read_csv(f"{base_url}/fire_stations.csv")
police_stations = pd.read_csv(f"{base_url}/police_stations.csv")
crash_data      = pd.read_csv(f"{base_url}/emergency_crash.csv")
road_nodes      = pd.read_csv(f"{base_url}/road_nodes.csv")
transport_data  = pd.read_csv(f"{base_url}/cleaned_transport_2025%20(2).csv")
pedestrian_data = pd.read_csv(f"{base_url}/pedestrian_data.csv")

r = requests.get(f"{base_url}/road_edges.zip")
z = zipfile.ZipFile(io.BytesIO(r.content))
road_edges = pd.read_csv(z.open(z.namelist()[0]))

# Clean column names
all_datasets = [hospitals, fire_stations, police_stations, crash_data,
                road_nodes, road_edges, transport_data, pedestrian_data]

for df in all_datasets:
    df.columns = df.columns.str.strip()

# Standardise lat/lon column names
for df in [hospitals, fire_stations, police_stations, crash_data, road_nodes]:
    df.rename(columns={'latitude': 'lat', 'longitude': 'lon'}, inplace=True)

# Convert coordinates to float
for df in [hospitals, fire_stations, police_stations, crash_data, road_nodes]:
    df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
    df['lon'] = pd.to_numeric(df['lon'], errors='coerce')

# Filter hospitals to emergency-capable only
emergency_hospitals = hospitals[
    hospitals['emergency'].str.lower() == 'yes'
].copy().reset_index(drop=True)

# Drop rows with missing coordinates
police_stations = police_stations.dropna(subset=['lat', 'lon']).reset_index(drop=True)

# Quality check - coordinate datasets
coord_datasets = {
    'Emergency hospitals' : emergency_hospitals,
    'Fire stations'       : fire_stations,
    'Police stations'     : police_stations,
    'Crash incidents'     : crash_data,
    'Road nodes'          : road_nodes,
}

print("=" * 65)
print("  DATASET QUALITY CHECK")
print("=" * 65)

for name, df in coord_datasets.items():
    nulls   = df[['lat', 'lon']].isnull().sum().sum()
    lat_rng = f"{df['lat'].min():.2f} → {df['lat'].max():.2f}"
    lon_rng = f"{df['lon'].min():.2f} → {df['lon'].max():.2f}"
    print(f"\n  {name}")
    print(f"    Rows        : {len(df):,}")
    print(f"    Null coords : {nulls}")
    print(f"    Lat range   : {lat_rng}")
    print(f"    Lon range   : {lon_rng}")

# Road edges
total_length = road_edges['length'].sum() if 'length' in road_edges.columns else None
print(f"\n  Road edges")
print(f"    Rows        : {len(road_edges):,}")
if total_length:
    print(f"    Total length: {total_length:,.1f} metres")

# Transport data
print(f"\n  Transport data")
print(f"    Rows        : {len(transport_data):,}")
print(f"    Columns     : {list(transport_data.columns)}")
print(f"    Null values : {transport_data.isnull().sum().sum()}")

# Pedestrian data
print(f"\n  Pedestrian data")
print(f"    Rows        : {len(pedestrian_data):,}")
print(f"    Columns     : {list(pedestrian_data.columns)}")
print(f"    Null values : {pedestrian_data.isnull().sum().sum()}")

# Summary
print("\n" + "=" * 65)
print("  SUMMARY")
print("=" * 65)
print(f"  Emergency hospitals  : {len(emergency_hospitals)}")
print(f"  Fire stations        : {len(fire_stations)}")
print(f"  Police stations      : {len(police_stations)}")
print(f"  Crash incidents      : {len(crash_data):,}")
print(f"  Road nodes           : {len(road_nodes):,}")
print(f"  Road edges           : {len(road_edges):,}")
print(f"  Transport records    : {len(transport_data):,}")
print(f"  Pedestrian records   : {len(pedestrian_data):,}")
print("=" * 65)

  DATASET QUALITY CHECK

  Emergency hospitals
    Rows        : 26
    Null coords : 0
    Lat range   : -38.36 → -37.65
    Lon range   : 144.70 → 145.35

  Fire stations
    Rows        : 205
    Null coords : 0
    Lat range   : -38.43 → -37.50
    Lon range   : 144.50 → 145.75

  Police stations
    Rows        : 124
    Null coords : 0
    Lat range   : -38.37 → -37.51
    Lon range   : 144.58 → 145.72

  Crash incidents
    Rows        : 194,352
    Null coords : 0
    Lat range   : -39.03 → -34.12
    Lon range   : 140.97 → 149.76

  Road nodes
    Rows        : 228,213
    Null coords : 0
    Lat range   : -38.49 → -37.44
    Lon range   : 144.46 → 145.91

  Road edges
    Rows        : 501,205
    Total length: 59,139,312.0 metres

  Transport data
    Rows        : 258,573
    Columns     : ['id', 'location_name', 'Latitude', 'Longitude', 'date', 'class', 'count']
    Null values : 540

  Pedestrian data
    Rows        : 734,064
    Columns     : ['sensor_id', 'date', 'hour

**Results:** All 8 datasets loaded and validated successfully.

**Service facility coverage** spans the greater Melbourne and surrounding Victorian region with zero null coordinates across all five coordinate-bearing datasets:
- **Emergency hospitals** - 26 facilities, latitude −38.36 to −37.65, longitude 144.70 to 145.35
- **Fire stations** - 205 stations, latitude −38.43 to −37.50, longitude 144.50 to 145.75
- **Police stations** - 124 stations, latitude −38.37 to −37.51, longitude 144.58 to 145.72

**Crash incident data** covers a much wider area across Victoria (194,352 records), ranging from latitude −39.03 to −34.12 and longitude 140.97 to 149.76 confirming statewide coverage well beyond the metropolitan facility boundaries.

**Road network** consists of 228,213 nodes and 501,205 edges with a total road length of approximately 59,139 km, ready for routing engine in Sprint 2.

**Transport data** contains 258,573 records across 7 columns including location coordinates, date, transport class, and passenger counts. 540 null values were detected and will need to be handled depending on downstream usage.

**Pedestrian data** is the largest dataset at 734,064 records with zero null values. It includes hourly pedestrian counts by sensor, congestion scores, and peak hour flags, useful for understanding foot traffic density around incident locations.

---
## 1.1 - Routing Engine Setup

The routing engine replaces the `route: PENDING` placeholder from Sprint 1 with real shortest-path calculations on the Melbourne road network. The graph is loaded here once and reused across every agent respond call.

Loading the graph inside each individual agent call would be extremely slow since the Melbourne network contains 228,000+ nodes. Loading once at startup means routing stays fast at dispatch time.

> Route calculation uses the A* algorithm via `connect_routing()`.
> Source: `routing_engine.py` - Dijkstra / A* routing engine built on OSMnx and NetworkX by Basil Behanan.

In [6]:
import sys, os
sys.path.insert(0, os.path.join(os.path.expanduser('~'), 'Downloads'))

from routing_engine import connect_routing, load_or_build_graph

GRAPH_PATH = os.path.join(os.path.expanduser('~'), 'Downloads', 'melbourne.graphml')

print("Loading Melbourne graph...")
G = load_or_build_graph(graph_path=GRAPH_PATH)

print(f" Routing engine loaded")
print(f"  Nodes : {G.number_of_nodes():,}")
print(f"  Edges : {G.number_of_edges():,}")

Loading Melbourne graph...
 Routing engine loaded
  Nodes : 228,702
  Edges : 502,181


---
## 2 - Emergency Types and Dispatch Table

The dispatch table maps every recognised emergency type to the correct combination of responding agents and a priority level. Priority 1 is critical, 2 is high, 3 is medium.

The central dispatch function in Section 7 reads this table at runtime to decide which agents to activate.

In [8]:
# Maps each emergency type to the agents that should respond and a priority level.
# The dispatch() function in Section 7 reads this table at runtime.

# Emergency Types & Dispatch Table

DISPATCH_TABLE = {
    "cardiac_arrest": {
        "agents": ["ambulance"],
        "priority": 1,
        "notes": "Medical emergency - ambulance only"
    },
    "fire": {
        "agents": ["fire", "ambulance"],
        "priority": 1,
        "notes": "Fire service leads, ambulance on standby for injuries"
    },
    "car_accident_minor": {
        "agents": ["ambulance", "police"],
        "priority": 2,
        "notes": "Ambulance for injuries, police for traffic control"
    },
    "car_accident_major": {
        "agents": ["ambulance", "fire", "police"],
        "priority": 1,
        "notes": "All services — entrapment likely, major injuries expected"
    },
    "robbery": {
        "agents": ["police"],
        "priority": 2,
        "notes": "Police only"
    },
    "assault": {
        "agents": ["police", "ambulance"],
        "priority": 2,
        "notes": "Police to secure scene, ambulance for victim"
    },
    "building_collapse": {
        "agents": ["fire", "ambulance", "police"],
        "priority": 1,
        "notes": "All services — search and rescue situation"
    },
    "gas_leak": {
        "agents": ["fire", "police"],
        "priority": 1,
        "notes": "Fire handles hazard, police for evacuation"
    },
    "drowning": {
        "agents": ["ambulance", "police"],
        "priority": 1,
        "notes": "Ambulance for resuscitation, police for scene control"
    },
    "unknown": {
        "agents": ["police"],
        "priority": 3,
        "notes": "Default - police assess and call in other services if needed"
    }
}

print(f"{'EMERGENCY TYPE':<25} {'AGENTS':<35} {'PRI'}")
print("─" * 65)
for etype, info in DISPATCH_TABLE.items():
    print(f"{etype:<25} {', '.join(info['agents']):<35} {info['priority']}")

EMERGENCY TYPE            AGENTS                              PRI
─────────────────────────────────────────────────────────────────
cardiac_arrest            ambulance                           1
fire                      fire, ambulance                     1
car_accident_minor        ambulance, police                   2
car_accident_major        ambulance, fire, police             1
robbery                   police                              2
assault                   police, ambulance                   2
building_collapse         fire, ambulance, police             1
gas_leak                  fire, police                        1
drowning                  ambulance, police                   1
unknown                   police                              3


**Results:** The dispatch table defines 10 emergency types across three priority levels. Six are classified as critical (priority 1): cardiac arrest, fire, major car accident, building collapse, gas leak, and drowning. Three are high priority (priority 2): minor car accident, robbery, and assault. One is medium priority (priority 3): unknown emergencies, which default to a police-first response for on scene assessment.

Agent combinations range from single-service responses (cardiac arrest dispatches ambulance only, robbery dispatches police only) to full three-service mobilisation for major incidents like building collapses and major car accidents. This ensures resource allocation scales proportionally to incident severity. The `unknown` type acts as a catch-all - police are sent first to assess the situation and can request additional services if needed.

---
## 3 - Nearest-Unit Lookup

Every agent needs to answer one question: given an incident at these coordinates, which is the closest facility to dispatch from?

`find_nearest()` uses the Haversine formula to calculate real surface distances between the incident and every candidate facility, then returns the closest match. The Haversine formula accounts for the curvature of the Earth subtracting coordinates directly gives the wrong answer at geographic scale.

`find_nearest_n()` returns the top n closest facilities, useful for identifying backup units.

In [11]:

# Haversine Distance & Nearest-Unit Functions

def haversine(lat1, lon1, lat2, lon2):
    """
    Great-circle distance between two lat/lon points.
    Returns distance in kilometres.
    """
    R    = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a    = (math.sin(dlat / 2) ** 2 +
            math.cos(math.radians(lat1)) *
            math.cos(math.radians(lat2)) *
            math.sin(dlon / 2) ** 2)
    return R * 2 * math.asin(math.sqrt(a))


def find_nearest(incident_lat, incident_lon, locations_df, name_col='name'):
    """
    Returns the single closest location from a DataFrame.

    Parameters:
    
    incident_lat, incident_lon : float
        Coordinates of the incident.
    locations_df : DataFrame
        Must contain 'lat', 'lon', and `name_col` columns.
    name_col : str
        Column used as the facility name in the result.

    Returns
    
    dict : name, lat, lon, distance_km
    """
    df = locations_df.copy()
    df['distance_km'] = df.apply(
        lambda row: haversine(incident_lat, incident_lon, row['lat'], row['lon']),
        axis=1
    )
    nearest = df.loc[df['distance_km'].idxmin()]
    return {
        'name'       : nearest[name_col],
        'lat'        : nearest['lat'],
        'lon'        : nearest['lon'],
        'distance_km': round(nearest['distance_km'], 3)
    }


def find_nearest_n(incident_lat, incident_lon, locations_df, n=3, name_col='name'):
    """Returns the n closest locations - useful for backup units."""
    df = locations_df.copy()
    df['distance_km'] = df.apply(
        lambda row: haversine(incident_lat, incident_lon, row['lat'], row['lon']),
        axis=1
    )
    top_n = df.nsmallest(n, 'distance_km')[[name_col, 'lat', 'lon', 'distance_km']]
    return top_n.reset_index(drop=True)


# Validation - Melbourne CBD test point 
TEST_LAT, TEST_LON = -37.8136, 144.9631

print("=" * 60)
print(f"  VALIDATION - Incident at Melbourne CBD ({TEST_LAT}, {TEST_LON})")
print("=" * 60)

for label, df in [('Emergency hospital', emergency_hospitals),
                  ('Fire station',       fire_stations),
                  ('Police station',     police_stations)]:
    result = find_nearest(TEST_LAT, TEST_LON, df)
    print(f"\n  Nearest {label}:")
    print(f"    {result['name']}  —  {result['distance_km']} km")

    top3 = find_nearest_n(TEST_LAT, TEST_LON, df, n=3)
    print(f"  Top 3:")
    for _, row in top3.iterrows():
        print(f"    {row['name']:<50} {row['distance_km']:.3f} km")

print("\n" + "=" * 60)

  VALIDATION - Incident at Melbourne CBD (-37.8136, 144.9631)

  Nearest Emergency hospital:
    St Vincent's Hospital  —  1.249 km
  Top 3:
    St Vincent's Hospital                              1.249 km
    The Royal Melbourne Hospital                       1.762 km
    St Vincent's Private Hospital East Melbourne       1.874 km

  Nearest Fire station:
    Fire Station No. 3  —  1.082 km
  Top 3:
    Fire Station No. 3                                 1.082 km
    Fire Station No. 2                                 1.125 km
    Fire Station No. 1                                 1.200 km

  Nearest Police station:
    Melbourne East Police Station  —  0.365 km
  Top 3:
    Melbourne East Police Station                      0.365 km
    Melbourne East Police Station                      0.435 km
    Melbourne Prosecutions                             0.450 km



**Results:** The `find_nearest()` function was validated using Melbourne CBD (−37.8136, 144.9631) as a test incident location. All three service types returned sensible nearest facilities with plausible distances:

- **Ambulance** - St Vincent's Hospital at 1.249 km, with The Royal Melbourne Hospital (1.762 km) and St Vincent's Private Hospital East Melbourne (1.874 km) as backup options. All three are within 2 km of the CBD, reflecting Melbourne's strong inner-city hospital coverage.
- **Fire** - Fire Station No. 3 at 1.082 km, with Stations No. 2 (1.125 km) and No. 1 (1.200 km) close behind. The tight clustering of the top 3 (all within ~120 metres of each other in distance ranking) suggests strong fire response coverage in the CBD.
- **Police** - Melbourne East Police Station at just 0.365 km, making it the closest facility of any service type. A duplicate entry appears in the top 3 at a slightly different distance (0.435 km), which may indicate two separate records for the same station or a nearby annex - worth investigating during data cleaning.

The Haversine-based lookup is functioning correctly across all three datasets and returning realistic distances for the Melbourne metropolitan area.

---
## 4 - Base Agent Class

All three agent types inherit from this base class. It handles the shared behaviour every agent needs: tracking whether a unit is available or busy, logging assignments with timestamps, and providing a standard respond() interface. This means we never duplicate the same logic across ambulance, fire and police.

In [14]:
# Base Agent Class

class Agent:
    """
    Base class for all emergency response agents.
    Ambulance, Fire, and Police agents inherit from this.
    """

    def __init__(self, agent_id, agent_type, location="DEPOT"):
        self.agent_id   = agent_id
        self.agent_type = agent_type
        self.location   = location
        self.status     = "available"
        self.log        = []

    def is_available(self):
        return self.status == "available"

    def assign(self, emergency):
        if not self.is_available():
            return f"{self.agent_id} is currently busy and cannot respond."
        self.status = "busy"
        timestamp   = datetime.now().strftime("%H:%M:%S")
        entry       = f"[{timestamp}] {self.agent_id} assigned to: {emergency}"
        self.log.append(entry)
        return entry

    def complete(self):
        self.status = "available"
        timestamp   = datetime.now().strftime("%H:%M:%S")
        entry       = f"[{timestamp}] {self.agent_id} is now available again."
        self.log.append(entry)
        return entry

    def respond(self, emergency_type, location, lat=None, lon=None):
        return {
            "agent_id"       : self.agent_id,
            "agent_type"     : self.agent_type,
            "emergency_type" : emergency_type,
            "destination"    : location,
            "status"         : "dispatched",
        }

    def get_log(self):
        print(f"\n── Activity Log: {self.agent_id} ──")
        if not self.log:
            print("  No activity yet.")
        for entry in self.log:
            print(f"  {entry}")

    def __repr__(self):
        return f"<Agent {self.agent_id} | type={self.agent_type} | status={self.status}>"


# Base class validation
test_agent = Agent("TEST_01", "ambulance")

print("=" * 60)
print("  BASE AGENT CLASS - VALIDATION")
print("=" * 60)
print(f"  Created        : {test_agent}")
print(f"  Is available?  : {test_agent.is_available()}")
print(f"  Assigning...   : {test_agent.assign('cardiac arrest at Main St')}")
print(f"  Is available?  : {test_agent.is_available()}")
print(f"  Status         : {test_agent.status}")
print(f"  Completing...  : {test_agent.complete()}")
print(f"  Is available?  : {test_agent.is_available()}")
print(f"  Status         : {test_agent.status}")
test_agent.get_log()
print("=" * 60)

  BASE AGENT CLASS - VALIDATION
  Created        : <Agent TEST_01 | type=ambulance | status=available>
  Is available?  : True
  Assigning...   : [01:31:41] TEST_01 assigned to: cardiac arrest at Main St
  Is available?  : False
  Status         : busy
  Completing...  : [01:31:41] TEST_01 is now available again.
  Is available?  : True
  Status         : available

── Activity Log: TEST_01 ──
  [01:31:41] TEST_01 assigned to: cardiac arrest at Main St
  [01:31:41] TEST_01 is now available again.


**Results:** The base `Agent` class is functioning correctly. The validation demonstrates the full lifecycle of an agent: creation in `available` status, assignment to an incident (which transitions the agent to `busy` and logs a timestamped entry), and completion (which returns the agent to `available`). The activity log correctly records both events in chronological order, confirming that the logging mechanism works as expected for downstream audit trails.

---
## 5 - Specialised Agents - Ambulance, Fire, Police

Each agent extends the base class with domain-specific logic and uses `find_nearest()` to resolve the closest real facility at dispatch time. Route calculation is handled by `connect_routing()` from the routing engine loaded in Section 1.1.

| Agent | Dataset | Nearest-unit lookup | Route |
|---|---|---|---|
| `AmbulanceAgent` | `emergency_hospitals` | Nearest emergency-capable hospital | A* via routing engine |
| `FireAgent` | `fire_stations` | Nearest CFA / MFB station | A* via routing engine |
| `PoliceAgent` | `police_stations` | Nearest police station | A* via routing engine |

In [17]:
# Ambulance, Fire and Police agents - each extends the base Agent class.
# find_nearest() resolves the closest real facility using Haversine distance.
# connect_routing() calculates the actual road route - powered by routing engine.


# Ambulance Agent

class AmbulanceAgent(Agent):
    """
    Handles medical emergencies.
    Resolves nearest emergency hospital from real dataset.
    Route calculated via Basil's routing engine.
    """

    ALS_REQUIRED = ["cardiac_arrest", "car_accident_major", "building_collapse", "drowning"]

    def __init__(self, agent_id, location="DEPOT_AMBULANCE"):
        super().__init__(agent_id, agent_type="ambulance", location=location)
        self.crew_size = 2

    def assess_severity(self, emergency_type):
        return "ALS" if emergency_type in self.ALS_REQUIRED else "BLS"

    def get_crew_size(self, care_level):
        return 3 if care_level == "ALS" else 2

    def respond(self, emergency_type, location, lat=None, lon=None):
        if not self.is_available():
            return {"agent_id": self.agent_id, "status": "unavailable",
                    "message": f"{self.agent_id} is busy — another unit must respond."}

        care_level = self.assess_severity(emergency_type)
        crew       = self.get_crew_size(care_level)

        if lat is not None and lon is not None:
            hospital_info    = find_nearest(lat, lon, emergency_hospitals)
            nearest_hospital = hospital_info['name']
            hospital_dist    = f"{hospital_info['distance_km']} km"
            route            = connect_routing(
                                    incident_lat=lat,
                                    incident_lon=lon,
                                    facility_lat=float(hospital_info['lat']),
                                    facility_lon=float(hospital_info['lon']),
                                    algorithm="astar",
                                    graph=G
                                )
        else:
            nearest_hospital = "unknown (no coordinates provided)"
            hospital_dist    = "unknown"
            route            = "No coordinates provided"

        self.assign(f"{emergency_type} at {location}")

        return {
            "agent_id"          : self.agent_id,
            "agent_type"        : "ambulance",
            "emergency_type"    : emergency_type,
            "destination"       : location,
            "coordinates"       : f"{lat}, {lon}" if lat else "not provided",
            "care_level"        : care_level,
            "crew_size"         : crew,
            "nearest_hospital"  : nearest_hospital,
            "hospital_distance" : hospital_dist,
            "status"            : "dispatched",
            "route"             : route
        }


# Fire Agent

class FireAgent(Agent):
    """
    Handles fire and hazardous material emergencies.
    Resolves nearest fire station from real dataset.
    Route calculated via Basil's routing engine.
    """

    EQUIPMENT_MAP = {
        "fire"              : ["hose_lines", "breathing_apparatus", "ladder_truck"],
        "car_accident_major": ["hydraulic_rescue_tools", "hose_lines"],
        "building_collapse" : ["search_rescue_kit", "breathing_apparatus", "hose_lines"],
        "gas_leak"          : ["hazmat_suit", "gas_detector", "breathing_apparatus"],
    }

    TRUCKS_NEEDED = {
        "fire": 2, "car_accident_major": 1, "building_collapse": 3, "gas_leak": 1,
    }

    def __init__(self, agent_id, location="DEPOT_FIRE"):
        super().__init__(agent_id, agent_type="fire", location=location)

    def get_equipment(self, emergency_type):
        return self.EQUIPMENT_MAP.get(emergency_type, ["standard_kit"])

    def get_trucks_needed(self, emergency_type):
        return self.TRUCKS_NEEDED.get(emergency_type, 1)

    def is_hazmat(self, emergency_type):
        return emergency_type == "gas_leak"

    def respond(self, emergency_type, location, lat=None, lon=None):
        if not self.is_available():\
            return {"agent_id": self.agent_id, "status": "unavailable",
                    "message": f"{self.agent_id} is busy — another unit must respond."}

        equipment = self.get_equipment(emergency_type)
        trucks    = self.get_trucks_needed(emergency_type)
        hazmat    = self.is_hazmat(emergency_type)

        if lat is not None and lon is not None:
            station_info    = find_nearest(lat, lon, fire_stations)
            nearest_station = station_info['name']
            station_dist    = f"{station_info['distance_km']} km"
            route           = connect_routing(
                                    incident_lat=lat,
                                    incident_lon=lon,
                                    facility_lat=float(station_info['lat']),
                                    facility_lon=float(station_info['lon']),
                                    algorithm="astar",
                                    graph=G
                                )
        else:
            nearest_station = "unknown (no coordinates provided)"
            station_dist    = "unknown"
            route           = "No coordinates provided"

        self.assign(f"{emergency_type} at {location}")

        return {
            "agent_id"         : self.agent_id,
            "agent_type"       : "fire",
            "emergency_type"   : emergency_type,
            "destination"      : location,
            "coordinates"      : f"{lat}, {lon}" if lat else "not provided",
            "trucks_dispatch"  : trucks,
            "equipment"        : equipment,
            "hazmat_situation" : hazmat,
            "nearest_station"  : nearest_station,
            "station_distance" : station_dist,
            "status"           : "dispatched",
            "route"            : route
        }


# Police Agent

class PoliceAgent(Agent):
    """
    Handles crime, crowd control, and scene security.
    Default first responder for unknown emergencies.
    Resolves nearest police station from real dataset.
    Route calculated via Basil's routing engine.
    """

    ARMED_RESPONSE_REQUIRED = ["robbery", "assault", "building_collapse", "unknown"]
    UNITS_NEEDED = {
        "robbery": 2, "assault": 2, "car_accident_minor": 1,
        "car_accident_major": 2, "building_collapse": 3,
        "gas_leak": 2, "drowning": 1, "unknown": 1,
    }
    CORDON_REQUIRED = ["car_accident_major", "building_collapse", "gas_leak", "fire"]

    def __init__(self, agent_id, location="DEPOT_POLICE"):
        super().__init__(agent_id, agent_type="police", location=location)
        self.is_lead = False

    def needs_armed_response(self, emergency_type):
        return emergency_type in self.ARMED_RESPONSE_REQUIRED

    def get_units_needed(self, emergency_type):
        return self.UNITS_NEEDED.get(emergency_type, 1)

    def needs_cordon(self, emergency_type):
        return emergency_type in self.CORDON_REQUIRED

    def set_as_lead(self):
        self.is_lead = True

    def respond(self, emergency_type, location, lat=None, lon=None):
        if not self.is_available():
            return {"agent_id": self.agent_id, "status": "unavailable",
                    "message": f"{self.agent_id} is busy — another unit must respond."}

        armed_response = self.needs_armed_response(emergency_type)
        units          = self.get_units_needed(emergency_type)
        cordon         = self.needs_cordon(emergency_type)

        if lat is not None and lon is not None:
            station_info    = find_nearest(lat, lon, police_stations)
            nearest_station = station_info['name']
            station_dist    = f"{station_info['distance_km']} km"
            route           = connect_routing(
                                    incident_lat=lat,
                                    incident_lon=lon,
                                    facility_lat=float(station_info['lat']),
                                    facility_lon=float(station_info['lon']),
                                    algorithm="astar",
                                    graph=G
                                )
        else:
            nearest_station = "unknown (no coordinates provided)"
            station_dist    = "unknown"
            route           = "No coordinates provided"

        self.assign(f"{emergency_type} at {location}")

        return {
            "agent_id"         : self.agent_id,
            "agent_type"       : "police",
            "emergency_type"   : emergency_type,
            "destination"      : location,
            "coordinates"      : f"{lat}, {lon}" if lat else "not provided",
            "units_dispatch"   : units,
            "armed_response"   : armed_response,
            "cordon_required"  : cordon,
            "lead_coordinator" : self.is_lead,
            "nearest_station"  : nearest_station,
            "station_distance" : station_dist,
            "status"           : "dispatched",
            "route"            : route
        }


# - Validation test ────────────────────────────
TEST_LAT, TEST_LON = -37.8136, 144.9631

amb  = AmbulanceAgent("AMB_TEST")
fire = FireAgent("FIRE_TEST")
pol  = PoliceAgent("POL_TEST")

test_cases = [
    (amb,  "cardiac_arrest",     "Flinders St Station"),
    (amb,  "car_accident_minor", "St Kilda Road"),
    (fire, "fire",               "Queen Victoria Market"),
    (fire, "gas_leak",           "South Melbourne"),
    (pol,  "robbery",            "Bourke Street Mall"),
    (pol,  "car_accident_major", "West Gate Freeway"),
]

print("=" * 65)
print("  SPECIALISED AGENTS — SPRINT 3 VALIDATION")
print("  Route: PENDING replaced with real routing engine")
print("  Test location: Melbourne CBD (-37.8136, 144.9631)")
print("=" * 65)

for agent, etype, loc in test_cases:
    agent.status = "available"
    agent.log    = []
    if hasattr(agent, 'is_lead'):
        agent.is_lead = False

    r = agent.respond(etype, loc, lat=TEST_LAT, lon=TEST_LON)
    print(f"\n  [{r['agent_type'].upper()}] {r['agent_id']} — {etype}")
    for k, v in r.items():
        if k not in ['agent_id', 'agent_type']:
            print(f"    {k:<24} : {v}")

print("\n" + "=" * 65)

  SPECIALISED AGENTS — SPRINT 3 VALIDATION
  Route: PENDING replaced with real routing engine
  Test location: Melbourne CBD (-37.8136, 144.9631)

  [AMBULANCE] AMB_TEST — cardiac_arrest
    emergency_type           : cardiac_arrest
    destination              : Flinders St Station
    coordinates              : -37.8136, 144.9631
    care_level               : ALS
    crew_size                : 3
    nearest_hospital         : St Vincent's Hospital
    hospital_distance        : 1.249 km
    status                   : dispatched
    route                    : {'status': 'SUCCESS', 'algorithm': 'astar', 'origin_node': 2190483586, 'destination_node': 6207152062, 'distance_m': 1413.3, 'distance_km': 1.413, 'estimated_travel_time_sec': 123.4, 'estimated_travel_time_min': 2.06, 'route_nodes': 25}

  [AMBULANCE] AMB_TEST — car_accident_minor
    emergency_type           : car_accident_minor
    destination              : St Kilda Road
    coordinates              : -37.8136, 144.9631
    c

**Results:** All three specialised agents were validated across six test scenarios using Melbourne CBD as the incident location.

**Ambulance agent** correctly differentiated between severity levels - cardiac arrest triggered Advanced Life Support (ALS) with a crew of 3, while a minor car accident triggered Basic Life Support (BLS) with a crew of 2. Both dispatches resolved St Vincent's Hospital as the nearest emergency-capable facility at 1.249 km, which is consistent with the `find_nearest()` validation in Section 3.

**Fire agent** adapted its response based on emergency type. A structural fire dispatched 2 trucks with standard firefighting equipment (hose lines, breathing apparatus, ladder truck) and no hazmat flag. A gas leak dispatched 1 truck with specialised hazmat gear (hazmat suit, gas detector, breathing apparatus) and correctly flagged `hazmat_situation: True`. Both resolved Fire Station No. 3 at 1.082 km.

**Police agent** correctly toggled between armed and unarmed responses - robbery triggered `armed_response: True` with 2 units, while a major car accident triggered `armed_response: False` but flagged `cordon_required: True` with 2 units. Both resolved Melbourne East Police Station at 0.365 km.

All six test cases returned real facility names and distances from the Victorian datasets, confirming that the placeholder data from Sprint 1 has been fully replaced.

---
## 6 - Area Context - Pedestrian and Transport Awareness

At dispatch time the system queries the pedestrian and transport datasets to build a picture of conditions around the incident. This gives responding units useful situational awareness before they arrive.

`get_area_context()` finds all sensors within 1km of the incident and returns:
- Pedestrian congestion level (LOW / MEDIUM / HIGH) based on real sensor scores
- Whether the area is in a peak hour window
- Dominant transport class nearby (car, cyclist, pedestrian)
- Names of nearby sensor locations


In [20]:
# Queries pedestrian and transport sensor data around the incident location.
# Returns a congestion and traffic summary used in the dispatch incident report.

def get_area_context(incident_lat, incident_lon, radius_km=1.0):
    """
    Returns a situational awareness summary for the area
    around an incident using pedestrian and transport data.
    """

    # Pedestrian context ─────────────────────
    ped = pedestrian_data.copy()
    ped['dist_km'] = ped.apply(
        lambda r: haversine(incident_lat, incident_lon,
                            r['Latitude'], r['Longitude']), axis=1
    )
    nearby_ped = ped[ped['dist_km'] <= radius_km]

    if not nearby_ped.empty:
        avg_congestion  = round(nearby_ped['congestion_score'].mean(), 3)
        peak_zone       = bool(nearby_ped['peak_hour'].mode()[0])
        max_pedestrians = int(nearby_ped['pedestrian_count'].max())

        # Thresholds based on actual distribution
        # Mean=1.0, 75th=1.25, Max=29.7
        if avg_congestion >= 2.0:
            congestion_level = "HIGH"
        elif avg_congestion >= 1.0:
            congestion_level = "MEDIUM"
        else:
            congestion_level = "LOW"
    else:
        avg_congestion   = None
        peak_zone        = False
        max_pedestrians  = 0
        congestion_level = "NO DATA"

    # Transport context ──────────────────────
    tpt = transport_data.copy()
    tpt.rename(columns={'Latitude': 'lat', 'Longitude': 'lon'}, inplace=True)
    tpt['dist_km'] = tpt.apply(
        lambda r: haversine(incident_lat, incident_lon,
                            r['lat'], r['lon']), axis=1
    )
    nearby_tpt = tpt[tpt['dist_km'] <= radius_km]

    if not nearby_tpt.empty:
        dominant_class   = nearby_tpt.groupby('class')['count'].sum().idxmax()
        total_activity   = int(nearby_tpt['count'].sum())
        sensor_locations = list(nearby_tpt['location_name'].unique()[:3])
    else:
        dominant_class   = "NO DATA"
        total_activity   = 0
        sensor_locations = []

    return {
        "radius_km"        : radius_km,
        "congestion_level" : congestion_level,
        "avg_congestion"   : avg_congestion,
        "peak_hour_zone"   : peak_zone,
        "max_pedestrians"  : max_pedestrians,
        "dominant_traffic" : dominant_class,
        "total_activity"   : total_activity,
        "nearby_sensors"   : sensor_locations,
    }


# Validation ────────────────────────────────
test_locations = [
    ("Melbourne CBD",   -37.8136, 144.9631),
    ("Fitzroy",         -37.7986, 144.9789),
    ("South Melbourne", -37.8319, 144.9548),
]

print("=" * 65)
print("  AREA CONTEXT - PEDESTRIAN & TRANSPORT AWARENESS")
print("  Search radius: 1.0 km | Thresholds: LOW<1.0, MEDIUM<2.0, HIGH≥2.0")
print("=" * 65)

for name, lat, lon in test_locations:
    ctx = get_area_context(lat, lon)
    print(f"\n  Location       : {name} ({lat}, {lon})")
    print(f"  Congestion     : {ctx['congestion_level']} "
          f"(score: {ctx['avg_congestion']})")
    print(f"  Peak hour zone : {ctx['peak_hour_zone']}")
    print(f"  Max pedestrians: {ctx['max_pedestrians']}")
    print(f"  Traffic type   : {ctx['dominant_traffic']} "
          f"({ctx['total_activity']} total activity)")
    if ctx['nearby_sensors']:
        print(f"  Nearby sensors : {', '.join(ctx['nearby_sensors'])}")

print("\n" + "=" * 65)

  AREA CONTEXT - PEDESTRIAN & TRANSPORT AWARENESS
  Search radius: 1.0 km | Thresholds: LOW<1.0, MEDIUM<2.0, HIGH≥2.0

  Location       : Melbourne CBD (-37.8136, 144.9631)
  Congestion     : MEDIUM (score: 1.288)
  Peak hour zone : False
  Max pedestrians: 9127
  Traffic type   : car (1262272 total activity)
  Nearby sensors : CoM pole 1112, Queens Bridge Street - CoM1549, Southbank Prom Asset ID: COM1598

  Location       : Fitzroy (-37.7986, 144.9789)
  Congestion     : NO DATA (score: None)
  Peak hour zone : False
  Max pedestrians: 0
  Traffic type   : car (49254 total activity)
  Nearby sensors : Drummond St - Reading Book Store

  Location       : South Melbourne (-37.8319, 144.9548)
  Congestion     : MEDIUM (score: 1.304)
  Peak hour zone : False
  Max pedestrians: 5633
  Traffic type   : NO DATA (0 total activity)



**Results:** Area context successfully generated for 3 Melbourne locations using real pedestrian and transport sensor data.

- **Melbourne CBD** - MEDIUM congestion (score 1.288), maximum 9,127 pedestrians recorded nearby, dominant traffic class is car with 1.26M total activity counts across nearby sensors. Not currently in a peak hour window.
- **South Melbourne** - MEDIUM congestion (score 1.304), maximum 5,633 pedestrians nearby, no transport sensor coverage in this area.
- **Fitzroy** - No pedestrian sensor coverage in the dataset. Transport data returns car activity from one nearby sensor. This is a data coverage limitation - the pedestrian sensor network is concentrated in the inner CBD.

**Known limitation:** The pedestrian sensor network covers Melbourne CBD and immediate surrounds. Incidents in suburbs like Fitzroy will return NO DATA for congestion level. This does not affect dispatch logic - area context is supplementary information only. Regional coverage can be expanded as additional sensor data becomes available.

**Integration:** `get_area_context()` is now called inside `dispatch()` and included as `area_context` in every incident report, giving responding units real situational awareness at dispatch time.

---
## 7 - Agent Pool and Central Dispatch Function

The single entry point for the entire system. Given an emergency type, location and coordinates, `dispatch()`:

1. Looks up the dispatch table to find which agents are needed
2. Selects the first available unit of each required type from the agent pool
3. Sets police as lead coordinator when multiple services respond
4. Calls respond() on each selected agent - triggering real nearest-unit lookups and route calculations
5. Calls get_area_context() to include situational awareness in the incident report
6. Returns a structured incident report containing all agent responses

In [23]:
# Starting with a small pool of 2 units per service - this gets expanded in Section 7.1.
# dispatch() is the main function called by LLM integration layer.

# Agent Pool & Dispatch Function

AGENT_POOL = {
    "ambulance": [AmbulanceAgent("AMB_01"), AmbulanceAgent("AMB_02")],
    "fire": [FireAgent("FIRE_01"), FireAgent("FIRE_02")],
    "police": [PoliceAgent("POL_01"), PoliceAgent("POL_02")],
}


def reset_agent_pool():
    """Reset all agents to available with clean logs."""
    for agents in AGENT_POOL.values():
        for agent in agents:
            agent.status = "available"
            agent.log = []
            if hasattr(agent, "is_lead"):
                agent.is_lead = False


def dispatch(emergency_type, location, lat=None, lon=None):
    """
    Main dispatch function.

    Parameters
    ----------
    emergency_type : str
        Must match a key in DISPATCH_TABLE. Falls back to 'unknown'.
    location : str
        Human-readable location description.
    lat, lon : float, optional
        Incident coordinates for real nearest-unit lookups.

    Returns
    -------
    dict
        Full incident report with all agent responses.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if emergency_type not in DISPATCH_TABLE:
        emergency_type = "unknown"

    dispatch_info = DISPATCH_TABLE[emergency_type]
    agents_needed = dispatch_info["agents"]
    priority = dispatch_info["priority"]
    notes = dispatch_info["notes"]

    responses = []
    agents_sent = []
    unavailable = []

    for agent_type in agents_needed:
        agent = None

        for available_agent in AGENT_POOL[agent_type]:
            if available_agent.is_available():
                agent = available_agent
                break

        if agent is None:
            unavailable.append(agent_type)
            continue

        if agent_type == "police" and len(agents_needed) > 1:
            agent.set_as_lead()

        response = agent.respond(emergency_type, location, lat=lat, lon=lon)

        responses.append(response)
        agents_sent.append(agent.agent_id)

    return {
        "incident_id": f"INC_{datetime.now().strftime('%H%M%S')}",
        "timestamp": timestamp,
        "emergency_type": emergency_type,
        "location": location,
        "coordinates": f"{lat}, {lon}" if lat is not None and lon is not None else "not provided",
        "area_context": get_area_context(lat, lon) if lat is not None and lon is not None else "No coordinates provided",
        "priority": priority,
        "dispatch_notes": notes,
        "agents_sent": agents_sent,
        "unavailable": unavailable if unavailable else "none",
        "total_responses": len(responses),
        "responses": responses,
    }


def print_incident_report(report):
    """Print a full incident report."""
    print("\n" + "=" * 65)
    print(f"  INCIDENT REPORT — {report['incident_id']}")
    print("=" * 65)
    print(f"  Timestamp      : {report['timestamp']}")
    print(f"  Emergency      : {report['emergency_type'].upper()}")
    print(f"  Location       : {report['location']}")
    print(f"  Coordinates    : {report['coordinates']}")
    print(f"  Area context   : {report['area_context']}")
    print(f"  Priority       : {report['priority']} (1 = critical)")
    print(f"  Notes          : {report['dispatch_notes']}")
    print(f"  Agents sent    : {', '.join(report['agents_sent'])}")
    print(f"  Unavailable    : {report['unavailable']}")
    print(f"  Total responses: {report['total_responses']}")
    print("\n  ── Agent Responses ──")

    for response in report["responses"]:
        print(f"\n  [{response['agent_type'].upper()}] {response['agent_id']}")

        for key, value in response.items():
            if key not in ["agent_id", "agent_type"]:
                print(f"    {key:<24} : {value}")

    print("=" * 65)


# Dispatch validation — 3 scenarios

CBD_LAT, CBD_LON = -37.8136, 144.9631

print("=" * 65)
print("  DISPATCH FUNCTION — VALIDATION")
print("  3 scenarios: single agent, two agents, all three agents")
print("=" * 65)

# Single agent - robbery: police only
reset_agent_pool()
print_incident_report(
    dispatch("robbery", "Bourke Street Mall", lat=CBD_LAT, lon=CBD_LON)
)

# Two agents - assault: police + ambulance
reset_agent_pool()
print_incident_report(
    dispatch("assault", "Federation Square", lat=CBD_LAT, lon=CBD_LON)
)

# All three agents - building collapse: fire + ambulance + police
reset_agent_pool()
print_incident_report(
    dispatch("building_collapse", "Docklands", lat=CBD_LAT, lon=CBD_LON)
)

  DISPATCH FUNCTION — VALIDATION
  3 scenarios: single agent, two agents, all three agents

  INCIDENT REPORT — INC_013307
  Timestamp      : 2026-05-03 01:33:05
  Emergency      : ROBBERY
  Location       : Bourke Street Mall
  Coordinates    : -37.8136, 144.9631
  Area context   : {'radius_km': 1.0, 'congestion_level': 'MEDIUM', 'avg_congestion': 1.288, 'peak_hour_zone': False, 'max_pedestrians': 9127, 'dominant_traffic': 'car', 'total_activity': 1262272, 'nearby_sensors': ['CoM pole 1112', 'Queens Bridge Street - CoM1549', 'Southbank Prom Asset ID: COM1598']}
  Priority       : 2 (1 = critical)
  Notes          : Police only
  Agents sent    : POL_01
  Unavailable    : none
  Total responses: 1

  ── Agent Responses ──

  [POLICE] POL_01
    emergency_type           : robbery
    destination              : Bourke Street Mall
    coordinates              : -37.8136, 144.9631
    units_dispatch           : 2
    armed_response           : True
    cordon_required          : False
    

**Results:** The dispatch function was validated across three scenarios of increasing complexity, all using Melbourne CBD coordinates.

**Single-agent dispatch (robbery)** - correctly dispatched only POL_01 with armed response enabled and 2 units. The `lead_coordinator` field is `False` because police are only designated as lead when multiple service types respond together. Melbourne East Police Station was resolved at 0.365 km.

**Two-agent dispatch (assault)** - correctly dispatched both POL_01 and AMB_01. Police was automatically designated as lead coordinator (`lead_coordinator: True`) since multiple services were involved. The ambulance assessed the assault as BLS (Basic Life Support) with a crew of 2, which is appropriate for a non-critical injury. Both agents resolved their nearest real facilities independently.

**Three-agent dispatch (building collapse)** - the most complex scenario, correctly mobilising all three services. Fire dispatched 3 trucks with search and rescue equipment, ambulance escalated to ALS with a crew of 3 (reflecting the critical nature of a collapse), and police dispatched 3 units with both armed response and scene cordoning enabled. Police was again correctly designated as lead coordinator. All three agents resolved real nearest facilities from the Victorian datasets, with no agents listed as unavailable.

Across all three scenarios, the dispatch function correctly scaled agent combinations, priority levels, and domain-specific logic based on the emergency type confirming the dispatch table, agent pool, and nearest-unit lookups are working together as intended.

---
## 7.1 - Expanded Agent Pool

The agent pool defines how many units of each service type are available for dispatch. Expanding from 2 to 5 units per service means the system can handle multiple overlapping incidents without flagging units as unavailable. This pool replaces the one defined in Section 7.

In [26]:
# Expanded pool - 5 units per service type instead of 2.
# This overwrites the AGENT_POOL defined in Section 7.

AGENT_POOL = {
    "ambulance": [
        AmbulanceAgent("AMB_01"), AmbulanceAgent("AMB_02"),
        AmbulanceAgent("AMB_03"), AmbulanceAgent("AMB_04"),
        AmbulanceAgent("AMB_05"),
    ],
    "fire": [
        FireAgent("FIRE_01"), FireAgent("FIRE_02"),
        FireAgent("FIRE_03"), FireAgent("FIRE_04"),
        FireAgent("FIRE_05"),
    ],
    "police": [
        PoliceAgent("POL_01"), PoliceAgent("POL_02"),
        PoliceAgent("POL_03"), PoliceAgent("POL_04"),
        PoliceAgent("POL_05"),
    ],
}

def reset_agent_pool():
    """Reset all agents to available with clean logs."""
    for agents in AGENT_POOL.values():
        for agent in agents:
            agent.status = "available"
            agent.log    = []
            if hasattr(agent, 'is_lead'):
                agent.is_lead = False

print("=" * 55)
print("  AGENT POOL — EXPANDED")
print("=" * 55)
for service, agents in AGENT_POOL.items():
    print(f"  {service.upper():<12} : {len(agents)} units — "
          f"{', '.join(a.agent_id for a in agents)}")
print("=" * 55)

  AGENT POOL — EXPANDED
  AMBULANCE    : 5 units — AMB_01, AMB_02, AMB_03, AMB_04, AMB_05
  FIRE         : 5 units — FIRE_01, FIRE_02, FIRE_03, FIRE_04, FIRE_05
  POLICE       : 5 units — POL_01, POL_02, POL_03, POL_04, POL_05


---
## 7.2 - Concurrent Incident Stress Test

Tests the system under concurrent load by firing 5 incidents without resetting the agent pool between calls. With 5 units per service type the system handles overlapping incidents without flagging any service as unavailable. This test would have exhausted the original 2-unit pool by the third incident.

In [28]:

# Fire 5 incidents back to back without resetting the pool.
# Demonstrates that the expanded pool handles concurrent load cleanly.


CBD_LAT, CBD_LON = -37.8136, 144.9631

concurrent_incidents = [
    ("cardiac_arrest",    "Flinders Street Station"),
    ("fire",              "Queen Victoria Market"),
    ("robbery",           "Bourke Street Mall"),
    ("car_accident_major","West Gate Freeway"),
    ("building_collapse", "Docklands"),
]

# Reset once at the start - do NOT reset between incidents
reset_agent_pool()

print("=" * 65)
print("  CONCURRENT INCIDENT STRESS TEST")
print("  5 incidents — pool not reset between dispatches")
print("=" * 65)
print(f"\n  {'INCIDENT':<25} {'AGENTS SENT':<30} {'UNAVAILABLE'}")
print("  " + "─" * 61)

for etype, loc in concurrent_incidents:
    report = dispatch(etype, loc, lat=CBD_LAT, lon=CBD_LON)
    agents_str = ", ".join(report["agents_sent"]) if report["agents_sent"] else "NONE"
    unavail    = ", ".join(report["unavailable"]) if report["unavailable"] != "none" else "none"
    print(f"  {etype:<25} {agents_str:<30} {unavail}")

print("\n" + "=" * 65)

# Count how many units are now busy
busy_count = {
    svc: sum(1 for a in agents if a.status == "busy")
    for svc, agents in AGENT_POOL.items()
}
print("  UNITS STATUS AFTER 5 CONCURRENT INCIDENTS")
print("  " + "─" * 61)
for svc, count in busy_count.items():
    total = len(AGENT_POOL[svc])
    avail = total - count
    print(f"  {svc.upper():<12} : {count} busy, {avail} still available")
print("=" * 65)

  CONCURRENT INCIDENT STRESS TEST
  5 incidents — pool not reset between dispatches

  INCIDENT                  AGENTS SENT                    UNAVAILABLE
  ─────────────────────────────────────────────────────────────
  cardiac_arrest            AMB_01                         none
  fire                      FIRE_01, AMB_02                none
  robbery                   POL_01                         none
  car_accident_major        AMB_03, FIRE_02, POL_02        none
  building_collapse         FIRE_03, AMB_04, POL_03        none

  UNITS STATUS AFTER 5 CONCURRENT INCIDENTS
  ─────────────────────────────────────────────────────────────
  AMBULANCE    : 4 busy, 1 still available
  FIRE         : 3 busy, 2 still available
  POLICE       : 3 busy, 2 still available


**Results:** Five concurrent incidents were dispatched without resetting the agent pool between calls, simulating real overlapping emergency load.

All 5 incidents were handled successfully with zero unavailable units across the full concurrent run. After all 5 dispatches:

- **Ambulance** - 4 of 5 units deployed, 1 still available
- **Fire** - 3 of 5 units deployed, 2 still available
- **Police** - 3 of 5 units deployed, 2 still available

With the original 2-unit pool, this same scenario would have exhausted ambulance units by the third incident and flagged unavailable on the fourth and fifth. The expanded pool of 5 units per service type handles this concurrent load cleanly while maintaining reserve capacity for additional incoming incidents.

The `building_collapse` incident - the most resource-intensive scenario, requiring all three services simultaneously was dispatched last and still found available units across all three service types, confirming the pool sizing is sufficient for realistic concurrent metropolitan load.

---
## 8 - Real Victorian Crash Incidents through Dispatch

Pulls real incidents directly from the Victorian crash dataset and runs them through the full dispatch pipeline. Crash severity is mapped to the system emergency types using `map_severity_to_emergency()` - a rule-based classifier that will be replaced by a natural language extraction layer in a future sprint.

| Crash Severity | Emergency Type | Agents |
|---|---|---|
| Fatal accident | `car_accident_major` | Ambulance + Fire + Police |
| Serious injury | `car_accident_major` | Ambulance + Fire + Police |
| Non-injury | `car_accident_minor` | Ambulance + Police |

In [31]:
# Samples 5 real Victorian crash incidents and dispatches them through the full pipeline.
# map_severity_to_emergency() is a temporary rule-based classifier - will be replaced
# by natural language LLM layer in a future sprint.


# Real Crash Incidents through Dispatch

def map_severity_to_emergency(row):
    """Maps crash dataset severity to dispatch emergency types."""
    severity = row['severity'].lower()
    inc_type = row['incident_type'].lower()

    if 'fatal' in severity:
        return 'car_accident_major'
    elif 'serious' in severity:
        return 'car_accident_major'
    elif 'struck pedestrian' in inc_type:
        return 'car_accident_major'
    else:
        return 'car_accident_minor'


# Sample 5 real incidents
fatal   = crash_data[crash_data['severity'] == 'Fatal accident'].sample(2, random_state=42)
serious = crash_data[crash_data['severity'] == 'Serious injury accident'].sample(2, random_state=42)
minor   = crash_data[crash_data['severity'] == 'Non injury accident'].sample(1, random_state=42)
sample  = pd.concat([fatal, serious, minor]).reset_index(drop=True)

print("=" * 65)
print("  REAL VICTORIAN CRASH INCIDENTS - DISPATCH TEST")
print("  5 incidents sampled across fatal, serious, and non-injury")
print("=" * 65)

for _, row in sample.iterrows():
    reset_agent_pool()

    emergency_type = map_severity_to_emergency(row)
    location       = f"Incident {row['incident_id']} ({row['incident_type']})"
    lat, lon       = row['lat'], row['lon']

    print(f"\n  ┌─ Source Record ─────────────────────────────────────")
    print(f"  │ ID       : {row['incident_id']}")
    print(f"  │ Date     : {row['date']}  Time: {row['time']}")
    print(f"  │ Type     : {row['incident_type']}")
    print(f"  │ Severity : {row['severity']}")
    print(f"  │ People   : {row['people_involved']}")
    print(f"  │ Coords   : {lat}, {lon}")
    print(f"  │ Mapped   : {emergency_type}")
    print(f"  └──────────────────────────────────────────────────────")

    report = dispatch(emergency_type, location, lat=lat, lon=lon)
    print_incident_report(report)

  REAL VICTORIAN CRASH INCIDENTS - DISPATCH TEST
  5 incidents sampled across fatal, serious, and non-injury

  ┌─ Source Record ─────────────────────────────────────
  │ ID       : T20210019823
  │ Date     : 2021-09-25  Time: 22:25:00
  │ Type     : Collision with a fixed object
  │ Severity : Fatal accident
  │ People   : 1
  │ Coords   : -36.71361, 146.41008
  │ Mapped   : car_accident_major
  └──────────────────────────────────────────────────────

  INCIDENT REPORT — INC_013646
  Timestamp      : 2026-05-03 01:36:40
  Emergency      : CAR_ACCIDENT_MAJOR
  Location       : Incident T20210019823 (Collision with a fixed object)
  Coordinates    : -36.71361, 146.41008
  Area context   : {'radius_km': 1.0, 'congestion_level': 'NO DATA', 'avg_congestion': None, 'peak_hour_zone': False, 'max_pedestrians': 0, 'dominant_traffic': 'NO DATA', 'total_activity': 0, 'nearby_sensors': []}
  Priority       : 1 (1 = critical)
  Notes          : All services — entrapment likely, major injuries exp

**Results:** Five real Victorian crash incidents were successfully dispatched - 2 fatal, 2 serious injury, and 1 non-injury spanning locations from regional Victoria to inner Melbourne.

**Fatal incidents** both mapped correctly to `car_accident_major` (priority 1) and triggered all three services. The first incident (T20210019823) occurred in regional Victoria near Wangaratta, which exposed an important finding: the nearest facilities were over 130 km away (Maroondah Hospital at 158.9 km, Healesville CFA at 131.3 km, Warburton Police Station at 132.2 km). This is expected given that the facility datasets are concentrated in the greater Melbourne area, and highlights the need for expanded regional coverage in future dataset iterations. The second fatal incident (T20160002505) occurred closer to Melbourne, with all facilities resolving within 6.3 km - a much more realistic response scenario.

**Serious injury incidents** also mapped to `car_accident_major` and dispatched all three services. Distances were reasonable - the closest facility across both incidents was Fire Station No. 12 at just 1.41 km. Both correctly triggered ALS care level with a crew of 3, scene cordoning, and police as lead coordinator.

**Non-injury incident** (T20220009177) mapped to `car_accident_minor` (priority 2) and dispatched only ambulance and police - no fire service, which is correct. The ambulance downgraded to BLS with a crew of 2, and police sent just 1 unit with no armed response or cordoning. Notably, Fitzroy Police Station resolved at only 0.131 km, and St Vincent's Hospital at 0.673 km, reflecting excellent inner-city coverage.

One data quality note: incident T20190004503 resolved to an "Unnamed Police Facility" at 2.463 km, suggesting some records in `police_stations.csv` have missing or incomplete name fields worth flagging for the dataset team to review.

All 5 incidents dispatched with zero failures and zero unavailable agents.

---
## 9 - Full System Test - All 10 Emergency Types

Runs all 10 emergency types through dispatch using Melbourne CBD as a common test point. Each scenario resets the agent pool to simulate independent incidents. Confirms every emergency type produces a valid response with the correct agent combination and priority level.

In [34]:
# Run every emergency type through the dispatch system.
# Pool resets between each scenario to simulate independent incidents.


# Full System Test — All 10 Emergency Types

# Melbourne CBD as common test location
CBD_LAT, CBD_LON = -37.8136, 144.9631

test_scenarios = [
    ("cardiac_arrest",      "Flinders St Station"),
    ("fire",                "Queen Victoria Market"),
    ("car_accident_minor",  "St Kilda Road"),
    ("car_accident_major",  "West Gate Freeway"),
    ("robbery",             "Bourke Street Mall"),
    ("assault",             "Federation Square"),
    ("building_collapse",   "Docklands"),
    ("gas_leak",            "South Melbourne"),
    ("drowning",            "St Kilda Beach"),
    ("unknown",             "Carlton Gardens"),
]

print("=" * 75)
print("  FULL SYSTEM TEST - ALL 10 EMERGENCY TYPES")
print("  Location: Melbourne CBD area")
print("=" * 75)
print(f"  {'EMERGENCY':<25} {'PRI':<5} {'AGENTS DISPATCHED':<35} {'STATUS'}")
print("  " + "─" * 71)

summary = []
for emergency_type, location in test_scenarios:
    reset_agent_pool()
    report     = dispatch(emergency_type, location, lat=CBD_LAT, lon=CBD_LON)
    agents_str = ", ".join(report["agents_sent"]) if report["agents_sent"] else "NONE"
    status     = "OK" if report["total_responses"] > 0 else "FAILED"
    summary.append((emergency_type, report["priority"], agents_str, status))
    print(f"  {emergency_type:<25} {report['priority']:<5} {agents_str:<35} {status}")

total    = len(summary)
ok       = sum(1 for s in summary if s[3] == "OK")
failed   = total - ok
critical = sum(1 for s in summary if s[1] == 1)

print("  " + "─" * 71)
print(f"\n  Total scenarios         : {total}")
print(f"  Successfully dispatched : {ok}/{total}")
print(f"  Failed dispatches       : {failed}")
print(f"  Critical (P1) incidents : {critical}")
print("=" * 75)

  FULL SYSTEM TEST - ALL 10 EMERGENCY TYPES
  Location: Melbourne CBD area
  EMERGENCY                 PRI   AGENTS DISPATCHED                   STATUS
  ───────────────────────────────────────────────────────────────────────
  cardiac_arrest            1     AMB_01                              OK
  fire                      1     FIRE_01, AMB_01                     OK
  car_accident_minor        2     AMB_01, POL_01                      OK
  car_accident_major        1     AMB_01, FIRE_01, POL_01             OK
  robbery                   2     POL_01                              OK
  assault                   2     POL_01, AMB_01                      OK
  building_collapse         1     FIRE_01, AMB_01, POL_01             OK
  gas_leak                  1     FIRE_01, POL_01                     OK
  drowning                  1     AMB_01, POL_01                      OK
  unknown                   3     POL_01                              OK
  ──────────────────────────────────────────

**Results:** All 10 emergency types dispatched successfully with zero failures.

The agent combinations match the dispatch table exactly:
- **Single-agent responses** - cardiac arrest (ambulance only), robbery (police only), and unknown (police only) each dispatched one service as expected.
- **Two-agent responses** - fire (fire + ambulance), minor car accident (ambulance + police), assault (police + ambulance), gas leak (fire + police), and drowning (ambulance + police) each dispatched the correct pair.
- **Three-agent responses** - major car accident and building collapse both mobilised all three services (ambulance, fire, police).

Priority levels are correctly assigned: 6 incidents at priority 1 (critical), 3 at priority 2 (high), and 1 at priority 3 (medium). The `unknown` type correctly defaults to the lowest priority with police-only response, acting as the system's catch-all for unrecognised emergency types.

10/10 scenarios passed with 0 failed dispatches, confirming full coverage of the dispatch table against real Victorian facility data.

---
## 10 - Routing Integration Verification

Verifies that the routing engine is fully connected and `route: PENDING` has been replaced with real shortest-path output from the Melbourne road network.

Each agent response should show distance in km, travel time in minutes and node count.

> Routing powered by Basil's `routing_engine.py` using the A* algorithm on the Melbourne OSMnx graph.

In [37]:
# 
# SPRINT 3 — ROUTING INTEGRATION VERIFICATION
# 

print("=" * 65)
print("  SPRINT 3 — ROUTING INTEGRATION TEST")
print("  Verifying route: PENDING is replaced with real routes")
print("=" * 65)

# Test with Flinders St Station — known Melbourne location
TEST_LAT = -37.8183
TEST_LON = 144.9671
TEST_LOCATION = "Flinders Street Station"

# Run a car_accident_major — triggers all 3 agents
reset_agent_pool()
report = dispatch("car_accident_major", TEST_LOCATION, lat=TEST_LAT, lon=TEST_LON)

print(f"\n  Incident    : {report['emergency_type'].upper()}")
print(f"  Location    : {report['location']}")
print(f"  Agents sent : {', '.join(report['agents_sent'])}")
print(f"\n  ── Route Verification ──")

all_routed = True
for r in report['responses']:
    route = r.get('route', {})
    agent = r['agent_id']
    if isinstance(route, dict) and route.get('status') == 'SUCCESS':
        print(f"\n  [{r['agent_type'].upper()}] {agent}")
        print(f"    Status        :  {route['status']}")
        print(f"    Algorithm     : {route['algorithm']}")
        print(f"    Distance      : {route['distance_km']} km")
        print(f"    Travel time   : {route['estimated_travel_time_min']} min")
        print(f"    Route nodes   : {route['route_nodes']}")
    elif isinstance(route, dict) and route.get('status') == 'NO_ROUTE':
        print(f"\n  [{r['agent_type'].upper()}] {agent} — ✗ NO_ROUTE: {route.get('error')}")
        all_routed = False
    else:
        print(f"\n  [{r['agent_type'].upper()}] {agent} — ✗ Still showing: {route}")
        all_routed = False

print("\n" + "=" * 65)
if all_routed:
    print("All agents returning real routes — PENDING fully replaced")
else:
    print("Some agents still have issues — check errors above")
print("=" * 65)

  SPRINT 3 — ROUTING INTEGRATION TEST
  Verifying route: PENDING is replaced with real routes

  Incident    : CAR_ACCIDENT_MAJOR
  Location    : Flinders Street Station
  Agents sent : AMB_01, FIRE_01, POL_01

  ── Route Verification ──

  [AMBULANCE] AMB_01
    Status        :  SUCCESS
    Algorithm     : astar
    Distance      : 1.673 km
    Travel time   : 2.33 min
    Route nodes   : 25

  [FIRE] FIRE_01
    Status        :  SUCCESS
    Algorithm     : astar
    Distance      : 1.465 km
    Travel time   : 1.92 min
    Route nodes   : 14

  [POLICE] POL_01
    Status        :  SUCCESS
    Algorithm     : astar
    Distance      : 0.0 km
    Travel time   : 0.0 min
    Route nodes   : 1

All agents returning real routes — PENDING fully replaced


---
## 11 - Summary

### Datasets Integrated

| Dataset | Records | Used By |
|---|---|---|
| `hospitals.csv` | 26 emergency-capable | `AmbulanceAgent` - nearest hospital by Haversine distance |
| `fire_stations.csv` | 205 stations | `FireAgent` - nearest fire station by Haversine distance |
| `police_stations.csv` | 124 stations | `PoliceAgent` - nearest police station by Haversine distance |
| `emergency_crash.csv` | 194,352 incidents | Real Victorian crash data for dispatch validation |
| `road_nodes.csv` | 228,213 nodes | Road network - loaded for routing engine |
| `road_edges.csv` | 501,205 edges | Road network - loaded for routing engine |
| `cleaned_transport_2025.csv` | 258,573 records | Transport sensor data for area context |
| `pedestrian_data.csv` | 734,064 records | Pedestrian congestion data for area context |

### Test Results

| Metric | Result |
|---|---|
| Emergency types covered | 10 / 10 |
| Full system test pass rate | 10 / 10 |
| Real crash incidents dispatched | 5 / 5 |
| Failed dispatches | 0 |
| Concurrent incidents (stress test) | 5 / 5, zero unavailable |
| Routing engine integrated | A* via `connect_routing()` |
| Area context integrated | Pedestrian + transport sensors |

### Key Observations

- **Inner-city coverage is strong.** All three services resolved nearest facilities within 1-7 km for Melbourne incidents.
- **Regional coverage is limited.** A fatal incident near Wangaratta resolved facilities over 130 km away - expanded regional data needed.
- **Data quality note.** One police station resolved as 'Unnamed Police Facility' - incomplete name field flagged for review.
- **Severity mapping is temporary.** The rule-based classifier will be replaced by a natural language LLM layer.
- **Area context coverage gap.** Pedestrian sensors are concentrated in Melbourne CBD - suburbs like Fitzroy return NO DATA.

### Remaining Items

| Item | Notes |
|---|---|

| `map_severity_to_emergency()` | Rule-based - to be replaced with natural language classification |
